# 🎙️ Podcast Clipping — End-to-End Pipeline

Pipeline otomatis satu atap untuk mengolah video podcast mentah menjadi potongan klip pendek bernilai viral tinggi.

**Alur Kerja Utama:**
1. **Step -2**: Ekstraksi & Preprocessing Audio (FFmpeg, Denoising, Normalisasi Laju Kenyaringan)
2. **Step -1**: Deteksi Suara (VAD) & Transkripsi Cerdas (OpenAI Whisper / MLX Whisper)
3. **Step 0 s.d. 8**: Pipeline NLP (Chunking, Ringkasan, Scene Detection, Relevancy, Penilaian Virality via Llama 3.2 3B, Perankingan)

## 📦 Imports & System Setup

In [ ]:
import os
import sys
import json
import time
import re
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Tuple
from pathlib import Path
import torch
import torchaudio
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import whisper
from mlx_lm import load, generate

# Pastikan folder kerja saat ini ada di sys.path agar modul lokal bisa di-import
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from audio_preprocessor import AudioPreprocessor
from audio_preprocessor.config import PreprocessorConfig, FFmpegConfig, DenoiseConfig, VADConfig

print("✅ Semua pustaka utama dan modul lokal berhasil dimuat!")

## 🗂️ Data Structures

In [ ]:
@dataclass
class Segment:
    id: int
    start: float
    end: float
    text: str

@dataclass
class Scene:
    start: float
    end: float
    duration: float
    text: str
    sentences: List[Segment]

print("Dataclasses Segment & Scene diinisialisasi!")

## 🔧 Helper Functions

In [ ]:
def split_scene_recursive(
    scene_segments: List[Segment], 
    embeddings_map: Dict[int, np.ndarray], 
    max_duration: float = 60.0
) -> List[List[Segment]]:
    """
    Recursively splits a scene if its duration exceeds the max_duration.
    The split point is chosen from grammatically valid points (ends with punctuation
    and next segment starts with capital letter). If none exist, falls back to
    the absolute lowest similarity point.
    """
    duration = scene_segments[-1].end - scene_segments[0].start
    if duration <= max_duration:
        return [scene_segments]
        
    if len(scene_segments) <= 1:
        return [scene_segments]
        
    lowest_sim = float("inf")
    split_idx = -1
    
    # 1. Cari titik potong yang memenuhi syarat tata bahasa (Grammatical Split Points)
    grammatical_indices = []
    for i in range(len(scene_segments) - 1):
        seg_a = scene_segments[i]
        seg_b = scene_segments[i+1]
        
        text_a_stripped = seg_a.text.strip()
        ends_with_punctuation = text_a_stripped and text_a_stripped[-1] in [".", "?", "!"]
        
        text_b_stripped = seg_b.text.strip()
        next_is_capitalized = len(text_b_stripped) > 0 and text_b_stripped[0].isupper()
        
        if ends_with_punctuation and next_is_capitalized:
            grammatical_indices.append(i)
            
    # 2. Jika ada opsi valid secara tata bahasa, pilih yang similarity-nya terendah
    if grammatical_indices:
        for i in grammatical_indices:
            seg_a = scene_segments[i]
            seg_b = scene_segments[i+1]
            emb_a = embeddings_map[seg_a.id].reshape(1, -1)
            emb_b = embeddings_map[seg_b.id].reshape(1, -1)
            sim = cosine_similarity(emb_a, emb_b)[0][0]
            if sim < lowest_sim:
                lowest_sim = sim
                split_idx = i
    else:
        # Fallback: jika tidak ada opsi tanda titik kalimat utuh, potong di similarity terendah secara murni
        for i in range(len(scene_segments) - 1):
            seg_a = scene_segments[i]
            seg_b = scene_segments[i+1]
            emb_a = embeddings_map[seg_a.id].reshape(1, -1)
            emb_b = embeddings_map[seg_b.id].reshape(1, -1)
            sim = cosine_similarity(emb_a, emb_b)[0][0]
            if sim < lowest_sim:
                lowest_sim = sim
                split_idx = i
                
    if split_idx == -1:
        return [scene_segments]
        
    left_part = scene_segments[:split_idx + 1]
    right_part = scene_segments[split_idx + 1:]
    
    return (split_scene_recursive(left_part, embeddings_map, max_duration) + 
            split_scene_recursive(right_part, embeddings_map, max_duration))

def get_mlx_model_path(model_name: str) -> str:
    if "/" in model_name:
        return model_name
    mapping = {
        "tiny": "mlx-community/whisper-tiny-mlx",
        "tiny.en": "mlx-community/whisper-tiny.en-mlx",
        "base": "mlx-community/whisper-base-mlx",
        "base.en": "mlx-community/whisper-base.en-mlx",
        "small": "mlx-community/whisper-small-mlx",
        "small.en": "mlx-community/whisper-small.en-mlx",
        "medium": "mlx-community/whisper-medium-mlx",
        "medium.en": "mlx-community/whisper-medium.en-mlx",
        "large": "mlx-community/whisper-large-v3-mlx",
        "turbo": "mlx-community/whisper-large-v3-turbo"
    }
    return mapping.get(model_name.lower(), f"mlx-community/whisper-{model_name}-mlx")

print("Fungsi pembantu siap!")

## ⚙️ Configuration

Tentukan berkas video input Anda di sini. Jika Anda hanya ingin menjalankan ulang pipeline NLP pada berkas JSON transkrip lama tanpa melakukan preprocessing video ulang, kosongkan saja variabel `VIDEO_PATH`.

In [ ]:
# ── Path input video mentah
VIDEO_PATH = "/Users/satriabaladewaharahap/Downloads/powell speech.mp4"

# ── Pengaturan Output & Model NLP
OUTPUT_DIR = "output"
ALPHA = 0.5

# ── Pengaturan Whisper Speech-to-Text
WHISPER_MODEL = "medium"   # Opsi model: tiny, base, small, medium, large, turbo
WHISPER_BACKEND = "mlx"    # Gunakan 'mlx' untuk Apple Silicon, atau 'openai' untuk PyTorch umum

print("Konfigurasi global telah diatur!")

## 🔌 Step -2 — Audio Extraction & Preprocessing

Mengekstrak audio dari video, membersihkan noise, dan mendeteksi aktivitas suara (VAD).

In [ ]:
if VIDEO_PATH and os.path.exists(VIDEO_PATH):
    print(f"⏳ Memulai Preprocessing Audio untuk: {VIDEO_PATH}…")
    
    # Bangun konfigurasi preprocessor secara manual
    config = PreprocessorConfig(
        output_dir=OUTPUT_DIR,
        log_level="INFO",
        ffmpeg=FFmpegConfig(sample_rate=16000, channels=1, codec="pcm_s16le"),
        denoise=DenoiseConfig(snr_clean_threshold=20.0, snr_mild_threshold=10.0),
        vad=VADConfig(threshold=0.5, min_speech_duration_ms=250, min_silence_duration_ms=500)
    )
    
    preprocessor = AudioPreprocessor(config)
    prep_result = preprocessor.process(VIDEO_PATH)
    
    AUDIO_PATH = prep_result.preprocessed_audio_path
    VAD_PATH = prep_result.vad_metadata_path
    
    print(f"✅ Preprocessing audio selesai!")
    print(f"   - Audio Bersih : {AUDIO_PATH}")
    print(f"   - Metadata VAD : {VAD_PATH}")
else:
    # Jika tidak ada video, asumsikan kita memakai berkas fallback default di folder output
    print("⚠️ VIDEO_PATH kosong atau tidak ditemukan. Melewati langkah preprocessing.")
    video_name = "FOMC Press Conference"
    AUDIO_PATH = os.path.join(OUTPUT_DIR, f"{video_name}_preprocessed.wav")
    VAD_PATH = os.path.join(OUTPUT_DIR, f"{video_name}_vad_metadata.json")

## 🎙️ Step -1 — Whisper Speech-to-Text Transcription

Melakukan transkripsi segmen bersuara secara akurat berdasarkan panduan VAD.

In [ ]:
if VIDEO_PATH and os.path.exists(VIDEO_PATH):
    print("⏳ Memulai transkripsi Whisper STT…")
    
    # Cek ketersediaan MLX
    has_mlx = False
    try:
        import mlx.core as mx
        import mlx_whisper
        has_mlx = True
    except ImportError:
        pass
        
    backend = WHISPER_BACKEND
    if backend == "mlx" and not has_mlx:
        print("⚠️ MLX-whisper tidak terinstal. Otomatis beralih ke PyTorch (OpenAI) backend.")
        backend = "openai"
        
    # Setup hardware device untuk PyTorch
    stt_device = "cpu"
    if backend == "openai":
        if torch.backends.mps.is_available():
            stt_device = "mps"
        elif torch.cuda.is_available():
            stt_device = "cuda"
            
    print(f"   - Backend  : {backend.upper()}")
    if backend == "openai":
        print(f"   - Hardware : {stt_device.upper()}")
        
    # 1. Load Whisper Model
    print(f"   📥 Loading model '{WHISPER_MODEL}'…")
    if backend == "mlx":
        mlx_model_path = get_mlx_model_path(WHISPER_MODEL)
        from mlx_whisper.transcribe import ModelHolder
        model = ModelHolder.get_model(mlx_model_path, mx.float16)
    else:
        model = whisper.load_model(WHISPER_MODEL, device=stt_device)
        
    # 2. Load Waveform & VAD JSON
    waveform, sample_rate = torchaudio.load(AUDIO_PATH)
    with open(VAD_PATH, "r", encoding="utf-8") as f:
        vad_data = json.load(f)
    speech_segments = vad_data.get("speech_segments", [])
    print(f"✅ Model termuat. Memproses {len(speech_segments)} segmen percakapan…")
    
    # 3. Transcribe segment-by-segment
    global_segments = []
    segment_id = 0
    
    for idx, seg in enumerate(speech_segments):
        start_time = seg["start"]
        end_time = seg["end"]
        
        # Slice waveform audio
        start_sample = int(start_time * sample_rate)
        end_sample = int(end_time * sample_rate)
        segment_audio = waveform[0, start_sample:end_sample].numpy()
        
        if len(segment_audio) < 160:
            continue
            
        if backend == "mlx":
            res = mlx_whisper.transcribe(
                segment_audio.astype("float32"),
                path_or_hf_repo=mlx_model_path,
                word_timestamps=False,
                verbose=False
            )
        else:
            res = model.transcribe(segment_audio, word_timestamps=False, fp16=False)
            
        # Simpan & konversi waktu
        for s in res.get("segments", []):
            global_segments.append({
                "id": segment_id,
                "start": round(start_time + s["start"], 3),
                "end": round(start_time + s["end"], 3),
                "text": s["text"].strip()
            })
            segment_id += 1
            
    # 4. Save to output JSON
    video_stem = Path(VIDEO_PATH).name
    if video_stem.endswith('.mp4'):
        video_stem = video_stem[:-4]
    elif video_stem.endswith('.wav'):
        video_stem = video_stem[:-4]
        
    TRANSCRIPT_PATH = os.path.join(OUTPUT_DIR, f'{video_stem}_transcript_segments.json')
    
    with open(TRANSCRIPT_PATH, 'w', encoding='utf-8') as f:
        json.dump(global_segments, f, indent=2, ensure_ascii=False)
        f.write('\n')
        
    print(f"✅ Transkripsi Whisper Selesai! Disimpan ke: {TRANSCRIPT_PATH}")
else:
    # Jika tidak ada video, gunakan berkas segment JSON yang sudah ditentukan pada konfigurasi default
    print("⚠️ VIDEO_PATH kosong atau tidak ditemukan. Menggunakan file transkrip default.")
    video_name = "FOMC Press Conference"
    TRANSCRIPT_PATH = os.path.join(OUTPUT_DIR, f"{video_name}_transcript_segments.json")

## 🔌 Step 0 — Load Transcripts & Configure NLP Device

In [ ]:
if not os.path.isfile(TRANSCRIPT_PATH):
    raise FileNotFoundError(f"Transcript file not found: {TRANSCRIPT_PATH}")
    
with open(TRANSCRIPT_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
    
if not raw_data:
    print("⚠️ Warning: Transcript file is empty.")
    scenes = []
    global_summary = ""
else:
    segments = [
        Segment(id=s["id"], start=s["start"], end=s["end"], text=s["text"])
        for s in raw_data
    ]
    print(f"✅ Loaded {len(segments)} segments.")

device = "cpu"
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
print(f"💻 Device configured: {device.upper()}")

## 🧠 Step 0.1 — Load NLP Models

In [ ]:
print("⏳ Loading NLP & Deep Learning models…")
model_load_start = time.perf_counter()

# 1. LLM Summarizer (Llama 3.2 3B via MLX)
print("  📥 Loading Llama-3.2-3B-Instruct-4bit via MLX...")
pegasus_model, pegasus_tokenizer = load("mlx-community/Llama-3.2-3B-Instruct-4bit")

# 2. SentenceTransformer Embedding Model
print("  📥 Loading SentenceTransformer model 'sentence-transformers/all-mpnet-base-v2'…")
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)
    
print(f"✅ Models loaded in {time.perf_counter() - model_load_start:.2f} s")

## ✂️ Step 1 — Chunking Transcript

In [ ]:
if raw_data:
    print("⏳ Splitting segments into semantic chunks (max 1024 tokens)…")
    chunks: List[List[Segment]] = []
    current_chunk: List[Segment] = []
    current_tokens = 0

    for seg in segments:
        seg_tokens = len(pegasus_tokenizer.encode(seg.text, add_special_tokens=False))
        if current_tokens + seg_tokens > 1024:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = [seg]
            current_tokens = seg_tokens
        else:
            current_chunk.append(seg)
            current_tokens += seg_tokens
            
    if current_chunk:
        chunks.append(current_chunk)
        
    print(f"✅ Created {len(chunks)} chunks.")

## 📝 Step 2 — Summarize Each Chunk (Llama 3.2 3B)

In [ ]:
if raw_data:
    print("⏳ Generating summaries for each chunk (Llama 3.2 3B)…")
    chunk_summaries: List[str] = []
    for idx, chunk in enumerate(chunks):
        chunk_text = " ".join([s.text for s in chunk])
        
        prompt = (
            "You are a professional editor. Meticulously summarize the following transcript chunk. "
            "Your summary must be highly factual, objective, and capture the main arguments or topics discussed. "
            "Do NOT assume or guess the names of any speakers. Only refer to them as 'the speaker', 'the host', or 'the guest' "
            "unless their specific names are explicitly written in the transcript text. "
            "Start the summary directly with the key points discussed. Do NOT use introductory phrases like "
            "'The podcast discusses', 'In this transcript', 'This chunk talks about', 'Key points from', or similar introductions.\n\n"
            f"Transcript:\n{chunk_text}\n\n"
            "Summary:"
        )
        
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = pegasus_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        summary = generate(
            pegasus_model,
            pegasus_tokenizer,
            prompt=formatted_prompt,
            max_tokens=150,
            verbose=False
        ).strip()
        
        chunk_summaries.append(summary)
        print(f"  📝 Chunk {idx+1}/{len(chunks)} Summary: {summary[:80]}…")

## 🌐 Step 3 — Generate Global Summary (Llama 3.2 3B)

In [ ]:
if raw_data:
    print("⏳ Generating overall global summary (Llama 3.2 3B)…")
    combined_chunk_summaries = " ".join(chunk_summaries)

    prompt = (
        "You are a professional editor. Synthesize and write a cohesive global summary of the transcript "
        "based on these chunk summaries. Focus on the main topics, final decisions or takeaways, and core themes discussed. "
        "Do NOT assume or guess the names of any speakers. Only refer to them as 'the speaker', 'the host', or 'the guest' "
        "unless their specific names are explicitly written in the text. "
        "Start the summary directly with the core points. Do NOT use introductory phrases like 'This video is about', "
        "'The summaries show', 'In this discussion', or similar introductions. Limit the summary to exactly 2-3 sentences.\n\n"
        f"Summaries:\n{combined_chunk_summaries}\n\n"
        "Global Summary:"
    )
    
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = pegasus_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    global_summary = generate(
        pegasus_model,
        pegasus_tokenizer,
        prompt=formatted_prompt,
        max_tokens=256,
        verbose=False
    ).strip()
    print(f"✅ Global Summary: {global_summary}")

## 🎬 Step 4 — Scene Detection (Semantic Segmentation dengan Grammatical Boundary Guard)

In [ ]:
if raw_data:
    print("⏳ Segmenting transcript into scenes based on semantic similarity & grammatical guard…")

    if len(segments) <= 1:
        print("  ℹ️ Single segment or empty transcript. Skipping semantic boundary detection.")
        if segments:
            scenes = [Scene(
                start=segments[0].start,
                end=segments[0].end,
                duration=segments[0].end - segments[0].start,
                text=segments[0].text,
                sentences=segments
            )]
        else:
            scenes = []
    else:
        # Embed all sentences
        sentences_text = [seg.text for seg in segments]
        embeddings = embedding_model.encode(sentences_text, show_progress_bar=False)
        
        # Build map of segment_id -> embedding for fast retrieval in recursive split
        embeddings_map = {seg.id: embeddings[i] for i, seg in enumerate(segments)}
        
        # Calculate cosine similarities between consecutive sentences
        emb_a = embeddings[:-1]
        emb_b = embeddings[1:]
        a_norm = emb_a / np.linalg.norm(emb_a, axis=1, keepdims=True)
        b_norm = emb_b / np.linalg.norm(emb_b, axis=1, keepdims=True)
        similarities = np.sum(a_norm * b_norm, axis=1)
        
        # Calculate dynamic threshold
        mean_sim = np.mean(similarities)
        std_sim = np.std(similarities)
        threshold = mean_sim - 0.5 * std_sim
        print(f"  📊 Cosine Similarity Mean: {mean_sim:.3f} | Std: {std_sim:.3f} | Threshold: {threshold:.3f}")
        
        # Grammatical Boundary Guard:
        # Adegan hanya boleh dipecah jika segment saat ini diakhiri titik (. ? !)
        # DAN segment berikutnya diawali dengan huruf KAPITAL (mencegah false punctuation jeda napas)
        scene_groups: List[List[Segment]] = []
        current_group = [segments[0]]
        
        for i in range(len(similarities)):
            seg_current = segments[i]
            seg_next = segments[i+1]
            
            text_stripped = seg_current.text.strip()
            ends_with_punctuation = text_stripped and text_stripped[-1] in [".", "?", "!"]
            
            next_text_stripped = seg_next.text.strip()
            next_is_capitalized = len(next_text_stripped) > 0 and next_text_stripped[0].isupper()
            
            if similarities[i] < threshold and ends_with_punctuation and next_is_capitalized:
                scene_groups.append(current_group)
                current_group = [seg_next]
            else:
                current_group.append(seg_next)
        if current_group:
            scene_groups.append(current_group)
            
        print(f"  🧩 Created {len(scene_groups)} initial scenes from boundaries.")
        
        # Batasi durasi adegan maksimal 60 detik secara rekursif (mematuhi Grammatical Guard)
        scenes: List[Scene] = []
        for group in scene_groups:
            split_groups = split_scene_recursive(group, embeddings_map, max_duration=60.0)
            
            for split_group in split_groups:
                start = split_group[0].start
                end = split_group[-1].end
                duration = end - start
                
                # Buang adegan < 10 detik agar klip memiliki makna dan konteks tanya-jawab yang utuh
                if duration < 10.0:
                    continue
                    
                scene_text = " ".join([s.text for s in split_group])
                scenes.append(Scene(
                    start=round(start, 2),
                    end=round(end, 2),
                    duration=round(duration, 2),
                    text=scene_text,
                    sentences=split_group
                ))
                
    print(f"✅ Generated {len(scenes)} valid scenes (duration 10s - 60s).")

## 📊 Step 5 — Relevancy Scoring

In [ ]:
if raw_data and scenes:
    print("⏳ Calculating semantic relevancy scores against global summary…")
    global_summary_emb = embedding_model.encode(global_summary, show_progress_bar=False).reshape(1, -1)
    scene_texts = [sc.text for sc in scenes]
    scene_embeddings = embedding_model.encode(scene_texts, show_progress_bar=False)
    
    # Calculate cosine similarities
    for idx, sc_emb in enumerate(scene_embeddings):
        sim = cosine_similarity(sc_emb.reshape(1, -1), global_summary_emb)[0][0]
        relevancy_score = float(np.clip(sim, 0.0, 1.0))
        object.__setattr__(scenes[idx], "relevancy_score", round(relevancy_score, 4))
    print("✅ Relevancy scoring completed!")

## 🔥 Step 6 — Virality Classification (Llama 3.2 3B Zero-Shot)

In [ ]:
if raw_data and scenes:
    print("⏳ Evaluating virality potential via Llama 3.2 3B Integer Prompt…")
    for idx, scene in enumerate(scenes):
        prompt = (
            "You are an expert social media editor. Analyze the video segment and rate its viral potential "
            "from 1 to 10 (1 = extremely boring/intro/routine callouts/filler, 10 = highly engaging hook/key quote/dramatic reveal).\n\n"
            "Examples:\n"
            "Transcript: \"Good day. It is an honor to be back at the Federal Reserve.\" -> Rating: 3\n"
            "Transcript: \"Thank you. Colby from NYT.\" -> Rating: 1\n"
            "Transcript: \"Fed funds rate at 3.5% to 3.75%\" -> Rating: 9\n\n"
            f"Transcript: \\\"{scene.text}\\\"\n"
            "Rating (Respond ONLY with a single integer between 1 and 10):"
        )
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = pegasus_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        res_text = generate(pegasus_model, pegasus_tokenizer, prompt=formatted_prompt, max_tokens=5, verbose=False).strip()
        
        score_match = re.search(r"\d+", res_text)
        try:
            rating = float(score_match.group(0)) if score_match else 5.0
            rating = np.clip(rating, 1.0, 10.0)
            score = rating / 10.0  # Petakan ke skala 0.1 s.d. 1.0
        except ValueError:
            score = 0.5
            
        label = 1 if score >= 0.5 else 0
        object.__setattr__(scene, "viral_label", label)
        object.__setattr__(scene, "viral_score", round(score, 4))
        
    print("✅ Virality classification completed!")

## 🏆 Step 7 — Virality Reranking (Filter Cascade)

In [ ]:
if raw_data and scenes:
    # Filter adegan survivor
    survivor_scenes = [sc for sc in scenes if getattr(sc, "viral_label", 0) == 1]
    
    if not survivor_scenes:
        print("  ⚠️ No scenes passed the binary virality filter. Falling back to ranking all scenes.")
        survivor_scenes = scenes
    else:
        print(f"  ⚡ Filter Cascade: {len(survivor_scenes)}/{len(scenes)} scenes passed to final selection.")
        
    print("✅ Virality reranking completed!")

## 🥇 Step 8 — Final Ranking & Output

In [ ]:
if raw_data and scenes:
    print("⏳ Calculating combined scores and sorting final rankings…")
    scene_list: List[Dict[str, Any]] = []

    # Masukkan adegan survivor ke dalam list perankingan akhir
    for idx, sc in enumerate(survivor_scenes):
        rel_score = getattr(sc, "relevancy_score", 0.0)
        vir_score = getattr(sc, "viral_score", 0.0)
        vir_label = getattr(sc, "viral_label", 0)
        
        comb_score = ALPHA * rel_score + (1.0 - ALPHA) * vir_score
        
        # Smart Punctuation Preview: 
        # Melewati tanda titik kalimat pembuka jika panjangnya di bawah 60 karakter
        # agar visualisasi Jupyter tidak terpotong pada basa-basi pembuka yang pendek.
        full_text = sc.text
        if len(full_text) <= 200:
            preview_text = full_text
        else:
            end_match = list(re.finditer(r"[.!?]\s", full_text[:200]))
            
            # Cari tanda titik kalimat yang berjarak > 60 karakter agar sapaan pendek dilewati
            best_cut = -1
            for m in reversed(end_match):
                if m.end() > 60:
                    best_cut = m.end()
                    break
                    
            if best_cut != -1:
                preview_text = full_text[:best_cut].strip()
            else:
                # Fallback: jika tanda titik terakhir posisinya <= 60 (terlalu pendek/basa-basi),
                # potong di spasi terdekat sebelum 200 karakter agar kalimat utama berikutnya ikut tercetak.
                cut_idx = full_text.rfind(" ", 0, 200)
                preview_text = full_text[:cut_idx] + "..." if cut_idx > 0 else full_text[:197] + "..."

        scene_list.append({
            "start": sc.start,
            "end": sc.end,
            "duration": sc.duration,
            "text": preview_text,
            "relevancy_score": round(rel_score, 4),
            "viral_label": vir_label,
            "viral_score": round(vir_score, 4),
            "combined_score": round(comb_score, 4)
        })

    # Sort rankings descending
    by_relevancy = sorted(scene_list, key=lambda x: x["relevancy_score"], reverse=True)[:5]
    by_virality = sorted(scene_list, key=lambda x: x["viral_score"], reverse=True)[:5]
    by_combined = sorted(scene_list, key=lambda x: x["combined_score"], reverse=True)[:5]

    output_data = {
        "global_summary": global_summary,
        "total_scenes": len(scenes),
        "rankings": {
            "by_relevancy": by_relevancy,
            "by_virality": by_virality,
            "by_combined": by_combined
        }
    }

    # Simpan ke folder output menggunakan nama dasar video
    output_dir = os.path.dirname(TRANSCRIPT_PATH) if os.path.dirname(TRANSCRIPT_PATH) else "output"
    audio_stem = Path(TRANSCRIPT_PATH).name
    if audio_stem.endswith("_transcript_segments.json"):
        base_name = audio_stem[:-25]  # Potong '_transcript_segments.json'
    else:
        base_name = Path(TRANSCRIPT_PATH).parent.name
        
    scene_rankings_path = os.path.join(output_dir, f"{base_name}_scene_rankings.json")

    with open(scene_rankings_path, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=2, ensure_ascii=False)
        f.write("\n")
        
    std_path = "scene_rankings.json"
    with open(std_path, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=2, ensure_ascii=False)
        f.write("\n")
        
    print(f"💾 Output tersimpan ke: {scene_rankings_path}")
    print(f"💾 Output tersimpan ke standard: {std_path}")

## 📈 Results Summary

In [ ]:
if raw_data and scenes:
    print('═' * 60)
    print(f'  ✅ Total scenes detected : {len(scenes)}')
    print(f'  📝 Global Summary        : {global_summary[:150]}…')
    print('═' * 60)

    print('\n🥇 TOP 5 — BY COMBINED SCORE (Relevance + Virality):')
    for i, s in enumerate(by_combined, 1):
        print(f'  {i}. [{s["start"]:.1f}s → {s["end"]:.1f}s | {s["duration"]:.1f}s] ')
        print(f'     relevancy={s["relevancy_score"]:.3f}  viral={s["viral_score"]:.3f}  combined={s["combined_score"]:.3f}')
        print(f'     💬 {s["text"]}')
        print()

    print('\n🥈 TOP 5 — BY RELEVANCY SCORE (Topic Grounding):')
    for i, s in enumerate(by_relevancy, 1):
        print(f'  {i}. [{s["start"]:.1f}s → {s["end"]:.1f}s | {s["duration"]:.1f}s] ')
        print(f'     relevancy={s["relevancy_score"]:.3f}  viral={s["viral_score"]:.3f}  combined={s["combined_score"]:.3f}')
        print(f'     💬 {s["text"]}')
        print()

    print('\n🔥 TOP 5 — BY VIRALITY SCORE (Engagement Potential):')
    for i, s in enumerate(by_virality, 1):
        print(f'  {i}. [{s["start"]:.1f}s → {s["end"]:.1f}s | {s["duration"]:.1f}s] ')
        print(f'     relevancy={s["relevancy_score"]:.3f}  viral={s["viral_score"]:.3f}  combined={s["combined_score"]:.3f}')
        print(f'     💬 {s["text"]}')
        print()

else:
    print("❌ Tidak ada data adegan yang bisa ditampilkan.")